# DUBAI TOWER — Multi-Floor (L03 + L04): Semantic Room Graph + GNN Prediction

Two stacked apartment floors joined into **one connected semantic graph**, then a GNN predicts
each room's type.

1. Each floor's room-type OBJs (`assets/obj/F03/`, `assets/obj/F04/`) → typed closed **cells**.
2. **Doors** (`aperture.obj`) per floor → intra-floor **circulation edges** (proximity match).
3. The two floors are joined through the **stairs / core** — each lower-floor stair cell links
   to the vertically-aligned upper-floor stair cell.
4. Combined graph → MSD CSV dataset → train a **GraphSAGE node classifier** → predict room types.

> Both floors are modelled in the same local space (same footprint, same Z), so each floor is
> processed independently and the upper floor is offset in Z **for display only**; the graph
> connection between floors is the stair/core link.

## 1. Imports

In [1]:
import os, time, random
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "vscode"

from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Face import Face
from topologicpy.Cell import Cell
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Graph import Graph
from topologicpy.PyG import PyG
from collections import Counter

random.seed(42); np.random.seed(42)
renderer = "vscode"

## 2. Configuration

In [2]:
from pathlib import Path

FLOOR_TAGS = ["F03", "F04"]          # bottom -> top
FLOOR_NAMES = ["Level 03", "Level 04"]

def _obj_root():
    for base in [Path.cwd(), *Path.cwd().parents]:
        p = base / "assets" / "obj"
        if (p / "F03" / "bedroom.obj").exists():
            return p
    raise FileNotFoundError("Could not find assets/obj/F03/bedroom.obj")
OBJ_ROOT = _obj_root()
FLOOR_DIRS = [OBJ_ROOT / t for t in FLOOR_TAGS]
DATASET_DIR = Path.cwd() / "Exports" / "Prediction_MultiFloor"
DATASET_DIR.mkdir(parents=True, exist_ok=True)
for t, d in zip(FLOOR_TAGS, FLOOR_DIRS):
    print(f"  {t}: {d}")
print("DATASET_DIR:", DATASET_DIR)

ROOM_FILES = {"bedroom":"bedroom.obj","livingroom":"livingroom.obj","kitchen":"kitchen.obj",
              "corridor":"corridor.obj","stairs":"stairs.obj","bathroom":"bathroom.obj",
              "storeroom":"storeroom.obj","balcony":"balcony.obj"}
DOOR_FILE, WINDOW_FILE = "aperture.obj", "window.obj"

ROOM_LABEL = {"bedroom":0,"livingroom":1,"kitchen":2,"dining":3,"corridor":4,
              "stairs":5,"storeroom":6,"bathroom":7,"balcony":8}
ZONING = {"bedroom":[1,0,0,0],"livingroom":[0,1,0,0],"kitchen":[0,1,0,0],"dining":[0,1,0,0],
          "corridor":[0,1,0,0],"stairs":[0,0,1,0],"storeroom":[0,0,1,0],"bathroom":[0,0,1,0],
          "balcony":[0,0,0,1]}
NODE_CONN = {"bedroom":[0,1,0],"livingroom":[0,1,0],"kitchen":[0,1,0],"dining":[0,1,0],
             "corridor":[1,0,0],"stairs":[1,0,0],"storeroom":[0,1,0],"bathroom":[0,1,0],
             "balcony":[0,1,0]}
DOOR_CONN  = [0,1,0]    # interior door
STAIR_CONN = [1,0,0]    # vertical stair/core link (passage)
ROOM_COLOR = {"bedroom":"#FFBFBF","bathroom":"#4444FF","corridor":"#7FFFBF","kitchen":"#BF3F3F",
              "livingroom":"#FFBF00","stairs":"#BF3FFF","storeroom":"#FF7FFF","dining":"#A0522D",
              "balcony":"#007F00","unknown":"#AAAAAA"}

PROXIMITY_TOL    = 0.6     # door/window -> room match distance (m)
STAIR_MATCH_TOL  = 6.0     # max XY distance to link a lower stair to an upper stair
DISPLAY_FLOOR_GAP = 40.0   # vertical separation between floors in the 3D view (display only)

  F03: c:\Users\Win11\GraphML_RaniaChihaoui\assets\obj\F03
  F04: c:\Users\Win11\GraphML_RaniaChihaoui\assets\obj\F04
DATASET_DIR: c:\Users\Win11\GraphML_RaniaChihaoui\DubaiTower\Exports\Prediction_MultiFloor


## 3. Process each floor independently (rooms → cells, doors → edges)

Each floor is loaded and matched in its own local space (the floors overlap in Z, so they must
not be mixed during door-matching).

In [3]:
def build_cell(faces):
    for tol in [0.001, 0.005, 0.01, 0.05, 0.1]:
        c = Cell.ByFaces(faces, tolerance=tol)
        if c is not None: return c
    return None

def load_faces(path):
    if not os.path.exists(path): return []
    objs = Topology.ByOBJPath(path, selfMerge=False)
    if not isinstance(objs, list): objs = [objs]
    faces = []
    for obj in objs:
        if obj is None: continue
        fs = Topology.Faces(obj) or []
        if fs: faces.extend(fs)
        else:
            for w in (Topology.Wires(obj) or []):
                f = Face.ByWire(w)
                if f is None:
                    w2 = Topology.RemoveCollinearEdges(w); f = Face.ByWire(w2) if w2 else None
                if f is not None: faces.append(f)
    return faces

def process_floor(floor_dir):
    cells, rtypes = [], []
    for rtype, fn in ROOM_FILES.items():
        path = str(floor_dir / fn)
        if not os.path.exists(path): continue
        objs = Topology.ByOBJPath(path, transposeAxes=True)
        if not isinstance(objs, list): objs = [objs]
        for obj in objs:
            if obj is None: continue
            faces = Topology.Faces(obj) or []
            if len(faces) < 4: continue
            c = build_cell(faces)
            if c is None: continue
            cells.append(c); rtypes.append(rtype)
    n = len(cells)
    cen = np.array([[Vertex.X(v), Vertex.Y(v), Vertex.Z(v)]
                    for v in (Topology.Centroid(c) for c in cells)])
    # doors -> intra-floor edges
    doors = load_faces(str(floor_dir / DOOR_FILE))
    def nearest(face, k):
        fc = Topology.Centroid(face)
        ds = sorted((Vertex.Distance(fc, cells[i]), i) for i in range(n))
        return [i for d, i in ds if d <= PROXIMITY_TOL][:k]
    eset = set()
    for d in doors:
        nb = nearest(d, 2)
        if len(nb) >= 2:
            a, b = sorted(nb[:2])
            if a != b: eset.add((a, b))
    # adjacency fallback: connect each still-isolated room (e.g. a balcony whose access
    # door isn't in aperture.obj) to its nearest room within the floor.
    _fdeg = Counter()
    for a, b in eset: _fdeg[a] += 1; _fdeg[b] += 1
    _c2 = cen[:, :2]
    for i in range(n):
        if _fdeg[i] == 0 and n > 1:
            d = ((_c2 - _c2[i])**2).sum(1); d[i] = 1e18
            j = int(d.argmin())
            if d[j]**0.5 <= 8.0: eset.add(tuple(sorted((i, j))))
    # windows -> per-room count
    wins = load_faces(str(floor_dir / WINDOW_FILE))
    wcount = [0]*n
    for w in wins:
        nb = nearest(w, 1)
        if nb: wcount[nb[0]] += 1
    return dict(cells=cells, rtypes=rtypes, cen=cen, edges=sorted(eset), wcount=wcount, n=n)

floors = []
for tag, d in zip(FLOOR_TAGS, FLOOR_DIRS):
    fl = process_floor(d)
    floors.append(fl)
    print(f"{tag}: {fl['n']} rooms, {len(fl['edges'])} edges  |  {dict(Counter(fl['rtypes']))}")

Cell.ByFaces - Warning: Could not construct cell from cleaned faces. Trying vertex-fused reconstruction.
Cell.ByFaces - Error: The operation failed. Returning None.
Cell.ByFaces - Warning: Could not construct cell from cleaned faces. Trying vertex-fused reconstruction.
Cell.ByFaces - Error: The operation failed. Returning None.
Cell.ByFaces - Warning: Could not construct cell from cleaned faces. Trying vertex-fused reconstruction.
Cell.ByFaces - Error: The operation failed. Returning None.
Cell.ByFaces - Warning: Could not construct cell from cleaned faces. Trying vertex-fused reconstruction.
Cell.ByFaces - Error: The operation failed. Returning None.
Cell.ByFaces - Warning: Could not construct cell from cleaned faces. Trying vertex-fused reconstruction.
Cell.ByFaces - Error: The operation failed. Returning None.
F03: 101 rooms, 102 edges  |  {'bedroom': 20, 'livingroom': 9, 'kitchen': 9, 'corridor': 15, 'stairs': 3, 'bathroom': 26, 'storeroom': 10, 'balcony': 9}
Cell.ByFaces - Warning

## 4. Combine floors into one graph + link through the stairs/core

In [4]:
# Global node arrays
g_rtype, g_floor, g_xy = [], [], []
g_win = []
offset = [0]
for fi, fl in enumerate(floors):
    g_rtype += fl["rtypes"]
    g_floor += [fi]*fl["n"]
    g_xy    += [(float(fl["cen"][i,0]), float(fl["cen"][i,1])) for i in range(fl["n"])]
    g_win   += fl["wcount"]
    offset.append(offset[-1] + fl["n"])
NTOT = len(g_rtype)

# intra-floor edges (reindex to global ids)
edges = []
for fi, fl in enumerate(floors):
    base = offset[fi]
    for a, b in fl["edges"]:
        edges.append((base+a, base+b, "door"))

# inter-floor stair/core links: each lower stair -> nearest upper stair by XY
stair_links = 0
for fi in range(len(floors)-1):
    lo, hi = floors[fi], floors[fi+1]
    lo_st = [offset[fi]+i   for i in range(lo["n"]) if lo["rtypes"][i]=="stairs"]
    hi_st = [offset[fi+1]+i for i in range(hi["n"]) if hi["rtypes"][i]=="stairs"]
    for a in lo_st:
        if not hi_st: break
        ax, ay = g_xy[a]
        b = min(hi_st, key=lambda j: (g_xy[j][0]-ax)**2 + (g_xy[j][1]-ay)**2)
        d = ((g_xy[b][0]-ax)**2 + (g_xy[b][1]-ay)**2)**0.5
        if d <= STAIR_MATCH_TOL:
            edges.append((a, b, "stair")); stair_links += 1

deg = Counter()
for a, b, _ in edges: deg[a]+=1; deg[b]+=1
iso = sum(1 for i in range(NTOT) if deg[i]==0)
print(f"Combined: {NTOT} rooms, {len(edges)} edges ({stair_links} stair links), isolated={iso}")
print("Per-floor rooms:", {FLOOR_NAMES[i]: floors[i]['n'] for i in range(len(floors))})

# contiguous labels for classes present
present = sorted(set(g_rtype), key=lambda t: ROOM_LABEL[t])
LOCAL = {t:i for i,t in enumerate(present)}; NAME={i:t for t,i in LOCAL.items()}
N_CLASS=len(present)
print("Classes:", LOCAL)

Combined: 228 rooms, 235 edges (3 stair links), isolated=0
Per-floor rooms: {'Level 03': 101, 'Level 04': 127}
Classes: {'bedroom': 0, 'livingroom': 1, 'kitchen': 2, 'corridor': 3, 'stairs': 4, 'storeroom': 5, 'bathroom': 6, 'balcony': 7}


## 5. Visualise the combined multi-floor room graph (3D, floors stacked)

In [5]:
# display z by floor
dz = [g_floor[i]*DISPLAY_FLOOR_GAP for i in range(NTOT)]
fig = go.Figure()
# edges
for kind, col, w in [("door","rgba(80,80,80,0.45)",1.2), ("stair","red",4)]:
    ex,ey,ez=[],[],[]
    for a,b,k in edges:
        if k!=kind: continue
        ex+=[g_xy[a][0],g_xy[b][0],None]; ey+=[g_xy[a][1],g_xy[b][1],None]; ez+=[dz[a],dz[b],None]
    fig.add_trace(go.Scatter3d(x=ex,y=ey,z=ez,mode="lines",
                  line=dict(color=col,width=w),hoverinfo="skip",
                  name=("stair links" if kind=="stair" else "doors"),
                  showlegend=(kind=="stair")))
# nodes by type
for t in present:
    idx=[i for i in range(NTOT) if g_rtype[i]==t]
    fig.add_trace(go.Scatter3d(x=[g_xy[i][0] for i in idx],y=[g_xy[i][1] for i in idx],
                  z=[dz[i] for i in idx],mode="markers",
                  marker=dict(size=4,color=ROOM_COLOR[t],line=dict(color="black",width=0.5)),
                  name=t,text=[f"{FLOOR_NAMES[g_floor[i]]} {t}" for i in idx],hoverinfo="text"))
fig.update_layout(title=dict(text="Multi-floor semantic room graph (L03 + L04 via stairs)",font=dict(color="black")),
    paper_bgcolor="white",
    scene=dict(aspectmode="data",
        xaxis=dict(color="black",title="x"),yaxis=dict(color="black",title="y"),
        zaxis=dict(color="black",title="z (floor, display)")),
    scene_camera=dict(projection=dict(type="orthographic"),eye=dict(x=1.4,y=1.4,z=1.0)),
    legend=dict(font=dict(color="black")),width=1200,height=800)
fig.show(renderer=renderer)

## 6. Export combined MSD dataset (one graph spanning both floors)

In [6]:
split=[""]*NTOT; cnt=Counter(); order=list(range(NTOT)); random.shuffle(order)
for i in order:
    cnt[g_rtype[i]]+=1; k=cnt[g_rtype[i]]%5
    split[i]="train" if k<3 else ("val" if k==3 else "test")

node_rows=[]
for i in range(NTOT):
    z=ZONING[g_rtype[i]]; c=NODE_CONN[g_rtype[i]]
    node_rows.append(dict(graph_id=0,node_id=i,label=LOCAL[g_rtype[i]],
        feat_zoning_type_0=z[0],feat_zoning_type_1=z[1],feat_zoning_type_2=z[2],feat_zoning_type_3=z[3],
        feat_connectivity_0=c[0],feat_connectivity_1=c[1],feat_connectivity_2=c[2],
        train_mask=int(split[i]=="train"),val_mask=int(split[i]=="val"),test_mask=int(split[i]=="test")))
pd.DataFrame(node_rows).to_csv(DATASET_DIR/"nodes.csv",index=False)

erows=[]
for a,b,kind in edges:
    cf = STAIR_CONN if kind=="stair" else DOOR_CONN
    for s,d in [(a,b),(b,a)]:
        erows.append(dict(graph_id=0,src_id=s,dst_id=d,
            feat_connectivity_0=cf[0],feat_connectivity_1=cf[1],feat_connectivity_2=cf[2]))
pd.DataFrame(erows).to_csv(DATASET_DIR/"edges.csv",index=False)
pd.DataFrame([dict(graph_id=0,num_nodes=NTOT)]).to_csv(DATASET_DIR/"graphs.csv",index=False)
print("Wrote dataset to",DATASET_DIR,"| split:",dict(Counter(split)),"| classes:",N_CLASS)

Wrote dataset to c:\Users\Win11\GraphML_RaniaChihaoui\DubaiTower\Exports\Prediction_MultiFloor | split: {'test': 44, 'train': 140, 'val': 44} | classes: 8


## 7. Load into PyG, train the GNN, evaluate

In [7]:
pyg = PyG.ByCSVPath(path=str(DATASET_DIR), level="node", task="classification",
                    graphLabelType="categorical", nodeLabelType="categorical", edgeLabelType="categorical")
s=pyg.Summary(); print(f"Loaded {s['num_graphs']} graph, {s['num_outputs']} classes, conv={s['conv']}")
pyg.SetHyperparameters(epochs=200, lr=0.01, dropout=0.2, early_stopping=True, early_stopping_patience=30)
t0=time.time(); pyg.Train(); print(f"Trained in {time.time()-t0:.1f}s")
metrics=pyg.Test()
print("Test metrics:")
for k,v in metrics.items(): print(f"  {k:16s}: {v:.4f}")
try: pyg.PlotHistory()
except Exception as e: print("PlotHistory skipped:",e)
try: pyg.PlotConfusionMatrix(split="test")
except Exception as e: print("Confusion skipped:",e)

Loaded 1 graph, 8 classes, conv=sage
Trained in 2.2s
Test metrics:
  test_accuracy   : 0.9545
  test_precision  : 0.9697
  test_recall     : 0.9545
  test_f1         : 0.9570


## 8. Predict every room + export predictions, true-vs-predicted view

In [8]:
report=pyg.Predict(split="all", return_probs=True)
pred=report["pred"]
gp=np.asarray(pred[0]) if isinstance(pred,(list,tuple)) else np.asarray(pred)
def to_class(v):
    a=np.squeeze(np.asarray(v)); return int(a) if a.ndim==0 else (int(np.argmax(a)) if a.size>1 else int(a[0]))
rows=[]
for i in range(NTOT):
    yp=to_class(gp[i]); yt=LOCAL[g_rtype[i]]
    rows.append(dict(node_id=i,floor=FLOOR_NAMES[g_floor[i]],room_type=g_rtype[i],
        y_true=yt,y_pred=yp,true_name=NAME[yt],pred_name=NAME.get(yp,"?"),split=split[i],
        x=round(g_xy[i][0],2),y=round(g_xy[i][1],2)))
pred_df=pd.DataFrame(rows); pred_df.to_csv(DATASET_DIR/"node_predictions.csv",index=False)
acc_all=(pred_df.y_true==pred_df.y_pred).mean()
acc_test=(pred_df[pred_df.split=="test"].y_true==pred_df[pred_df.split=="test"].y_pred).mean()
print(f"Accuracy — all: {acc_all:.3f} | test: {acc_test:.3f}")
print("\nPer-floor accuracy:")
for fn in FLOOR_NAMES:
    sub=pred_df[pred_df.floor==fn]
    print(f"  {fn}: {(sub.y_true==sub.y_pred).mean():.3f}  ({len(sub)} rooms)")

# true vs predicted, panels per floor (rows) x TRUE/PREDICTED (cols)
nF=len(FLOOR_NAMES)
from plotly.subplots import make_subplots
fig=make_subplots(rows=nF,cols=2,horizontal_spacing=0.06,vertical_spacing=0.08,
                  subplot_titles=[f"{FLOOR_NAMES[f]} — {lbl}" for f in range(nF) for lbl in ("TRUE","PREDICTED")])
for fidx in range(nF):
    for col,name_key in [(1,"true_name"),(2,"pred_name")]:
        ex,ey=[],[]
        for a,b,k in edges:
            if g_floor[a]==fidx and g_floor[b]==fidx:
                ex+=[g_xy[a][0],g_xy[b][0],None]; ey+=[g_xy[a][1],g_xy[b][1],None]
        fig.add_trace(go.Scatter(x=ex,y=ey,mode="lines",line=dict(color="rgba(80,80,80,0.4)",width=1),
                      hoverinfo="skip",showlegend=False),row=fidx+1,col=col)
        for t in present:
            idx=[i for i in range(NTOT) if g_floor[i]==fidx and pred_df.iloc[i][name_key]==t]
            if not idx: continue
            wrong=[i for i in idx if pred_df.iloc[i].y_true!=pred_df.iloc[i].y_pred]
            fig.add_trace(go.Scatter(x=[g_xy[i][0] for i in idx],y=[g_xy[i][1] for i in idx],mode="markers",
                marker=dict(size=9,color=ROOM_COLOR[t],
                    line=dict(color=["red" if i in wrong else "black" for i in idx],
                              width=[3 if i in wrong else 1 for i in idx])),
                name=t,showlegend=(fidx==0 and col==1),
                text=[f"{pred_df.iloc[i].true_name}->{pred_df.iloc[i].pred_name}" for i in idx],
                hoverinfo="text"),row=fidx+1,col=col)
fig.update_xaxes(color="black"); fig.update_yaxes(color="black")
for ann in fig.layout.annotations: ann.font.color="black"
fig.update_layout(title=dict(text="True vs Predicted room types per floor (red ring = wrong)",font=dict(color="black")),
                  paper_bgcolor="white",plot_bgcolor="white",legend=dict(font=dict(color="black")),
                  width=1300,height=420*nF)
fig.show(renderer=renderer)

Accuracy — all: 0.925 | test: 0.955

Per-floor accuracy:
  Level 03: 0.931  (101 rooms)
  Level 04: 0.921  (127 rooms)


## 9. Summary

- Two floors (L03 + L04) processed independently, joined through the **stairs/core** into one
  connected semantic graph.
- Combined dataset (MSD schema) → **GraphSAGE** node classifier predicts room types across both
  floors; per-floor accuracy reported.

Artifacts in `dataset_multifloor/`. More data than single-floor → typically a sturdier model.
Use `pyg.CrossValidate(k_folds=5)` for a robust estimate, or `pyg.SaveModel(...)` to reuse it.

In [9]:
print("Artifacts:", DATASET_DIR)
for f in ["nodes.csv","edges.csv","graphs.csv","node_predictions.csv"]:
    p=DATASET_DIR/f; print(f"  {'OK ' if p.exists() else 'MISSING '}{f}")

Artifacts: c:\Users\Win11\GraphML_RaniaChihaoui\DubaiTower\Exports\Prediction_MultiFloor
  OK nodes.csv
  OK edges.csv
  OK graphs.csv
  OK node_predictions.csv
